In [15]:
!curl -L -o ~/Downloads/student-demographics-online-education-dataoulad.zip \
  "https://www.kaggle.com/api/v1/datasets/download/anlgrbz/student-demographics-online-education-dataoulad"

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0 42.1M    0  1177    0     0   2688      0  4:34:02 --:--:--  4:34:02  2688
curl: (23) Failure writing output to destination


In [ ]:
import os
import pandas as pd
import kagglehub

# 1. Download dataset automatically via kagglehub & load dynamically
path = kagglehub.dataset_download("anlgrbz/student-demographics-online-education-dataoulad")
print("Dataset path:", path)

files = os.listdir(path)
csv_filename = next((f for f in files if 'studentinfo' in f.lower()), files[0])
csv_file_path = os.path.join(path, csv_filename)

print(f"Loading file from: {csv_file_path}")
df = pd.read_csv(csv_file_path)

In [ ]:
import os
import pandas as pd

# Path to your static directory where the dataset should be placed
# If you are inside a Django environment, you can also use settings.STATIC_ROOT or BASE_DIR
data_dir = '/home/ouled/mysite/static'

# Check if directory exists
if os.path.exists(data_dir):
    files = os.listdir(data_dir)
    # Find the student CSV or JSON file locally
    csv_filename = next((f for f in files if 'studentinfo' in f.lower() or 'student_info' in f.lower()), None)
    
    if csv_filename:
        csv_file_path = os.path.join(data_dir, csv_filename)
        print(f"Loading file from: {csv_file_path}")
        df = pd.read_csv(csv_file_path)
    else:
        print("Target student data file not found in the static directory.")
else:
    print(f"Directory {data_dir} does not exist.")


Error: Could not find studentInfo CSV file in /home/ouled/mysite/data. Please upload it manually.


In [18]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import xgboost as xgb

In [19]:
pip install kagglehub pandas

# 1. Download latest version of the dataset from Kaggle automatically

In [ ]:
import os
import pandas as pd

# 1. Define your local data directory path
data_dir = '/home/ouled/mysite/data'

# 2. List files in the data directory to find the exact filename
files = os.listdir(data_dir)
print("Files available:", files)

# 3. Dynamically find the student info CSV file locally
csv_filename = next((f for f in files if 'studentInfo' in f.lower() or 'student_info' in f.lower()), files[0] if files else None)

if csv_filename:
    csv_file_path = os.path.join(data_dir, csv_filename)
    print(f"Loading file from: {csv_file_path}")

    # 4. Read with pandas
    df = pd.read_csv(csv_file_path)
    print("Successfully loaded DataFrame! Shape:", df.shape)
    print(df.head())
else:
    print(f"Error: No files found in {data_dir}. Please make sure you have uploaded your dataset CSV file there.")

Files available: ['processed_telemetry.json']
Loading file from: /home/ouled/mysite/data/processed_telemetry.json


In [ ]:
# 1. Download dataset automatically via kagglehub & load dynamically
path = kagglehub.dataset_download("anlgrbz/student-demographics-online-education-dataoulad")
print("Dataset path:", path)

files = os.listdir(path)
csv_filename = next((f for f in files if 'studentinfo' in f.lower()), files[0])
csv_file_path = os.path.join(path, csv_filename)

print(f"Loading file from: {csv_file_path}")
df = pd.read_csv(csv_file_path)

In [ ]:
# 2. Preprocessing & Feature Engineering
features = ['num_of_prev_attempts', 'studied_credits']
X = df[features].fillna(0)
y = df['final_result'].apply(lambda val: 1 if val in ['Pass', 'Distinction'] else 0)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
# 3. Dimensionality Reduction & Unsupervised Clustering (PCA & K-Means)
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_scaled)

In [ ]:
# 4. Supervised Training Split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
import xgboost as xgb

# Streamlined algorithm registry (faster execution)
models = {
    "Logistic Regression": LogisticRegression(),
    "Random Forest": RandomForestClassifier(n_estimators=50), # reduced estimators for speed
    "Naive Bayes": GaussianNB(),
    "k-Nearest Neighbors": KNeighborsClassifier(),
    "XGBoost": xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss')
}

results = {}
for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)
    results[name] = model.score(X_test, y_test)

print("\nModel Performance Summary:")
for model_name, score in results.items():
    print(f"- {model_name}: {score:.4f}")

In [ ]:
# 6. Export processed features/cluster metrics for Django consumption
import os

output_dir = '/home/ouled/mysite/data'
os.makedirs(output_dir, exist_ok=True)

id_col = 'id_student' if 'id_student' in df.columns else df.columns[0]

output_data = df[[id_col, 'code_module', 'final_result']].copy()
output_data['cluster'] = cluster_labels

output_path = os.path.join(output_dir, 'processed_telemetry.json')
output_data.to_json(output_path, orient='records')

print(f"\nSuccessfully exported telemetry metrics to '{output_path}'!")